# 실습 - LoRA

**원본 파라미터를 유지한 채 소형 조정 모듈만 학습**

- 원본 블록 — 전부 동결
- 각 블록 내부의 특정 가중치에 소형 모듈 삽입
- 학습 대상 — 추가한 모듈과 분류 헤드

**앞 실습(부분 미세조정)과의 차이**

| 구분 | 그대로 사용 | 달라지는 부분 |
|---|---|---|
| 데이터셋 | 동일한 구간 분리 결과 | — |
| 전처리 | 동일한 processor | — |
| 학습 도구 | Trainer, 동일한 절차 | — |
| 모델 구성 | — | LoRA 모듈 삽입 |
| 학습 범위 | — | 헤드 + 백본 내부 모듈 |
| 동결 코드 | — | 불필요 (자동 처리) |

**실습 순서**

- 데이터 · 전처리 준비 (앞 실습과 동일)
- 모델 재구성
- LoRA 설정 및 적용
- 학습 대상 확인
- 학습 실행
- 방식별 결과 비교

---
## 0. 환경 준비

- `peft` — PEFT 기법을 제공하는 전용 라이브러리
- LoRA · 어댑터 · 프롬프트 튜닝 등 지원

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 · 전처리 준비

- 앞 실습과 **동일한 코드**
- 앞 실습에 이어서 진행하는 경우 이 절은 건너뛴다
- 새로 시작하는 경우에만 실행

In [ ]:
ds = load_dataset("ethz/food101", split="train[:5000]")
ds = ds.train_test_split(test_size=0.2, seed=42)

labels = ds["train"].features["label"].names
id2label = {i: n for i, n in enumerate(labels)}
label2id = {n: i for i, n in enumerate(labels)}

ckpt = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(ckpt)

def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    out = processor(images, return_tensors="pt")
    out["labels"] = batch["label"]
    return out

ds = ds.with_transform(transform)

print("범주 수:", len(labels))

---
## 2. 모델 재구성

- 앞 실습에서 백본을 동결하고 헤드를 학습한 상태
- 비교를 위해 **학습되지 않은 모델**에서 다시 시작
- 동일 조건에서 두 방식의 결과 비교

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    ckpt,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

print(model.classifier)

**삽입 위치 확인**

- `target_modules` 에 지정할 이름은 모델마다 상이
- 아래 출력에서 어텐션 내부의 모듈 이름을 확인

In [ ]:
for name, _ in model.named_modules():
    if name.endswith(("query", "value")):
        print(name)
        break

---
## 3. LoRA 설정 및 적용

**주요 항목**

- `r` — 추가 모듈의 규모, 클수록 표현력 · 파라미터 증가
- `target_modules` — 삽입 위치, 어텐션의 query · value
- `lora_alpha` — 추가분의 반영 강도
- `lora_dropout` — 과적합 방지 비율
- `modules_to_save` — LoRA 외에 함께 학습할 부분

**주의** — 분류 헤드는 새로 생성되었으므로 `modules_to_save` 에 **반드시 포함**한다. 누락 시 헤드가 학습되지 않아 정확도가 오르지 않으며, 오류도 발생하지 않아 원인 파악이 어렵다.

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, config)

print("LoRA 적용 완료")

`get_peft_model` 은 원본 모델에 LoRA 모듈을 삽입한 형태로 변환한다. 원본 파라미터는 자동으로 동결되므로 별도의 동결 코드가 필요하지 않다.

---
## 4. 학습 대상 확인

- `print_trainable_parameters` — 학습 대상 파라미터 수와 비율 출력
- 학습 대상 — 삽입한 LoRA 모듈 + 분류 헤드
- 원본 백본은 동결 상태

**방식별 비교**

| 방식 | 학습 대상 | 비율 |
|---|---|---|
| 전체 미세조정 | 전체 | 100% |
| 부분 미세조정 | 헤드만 | 약 0.09% |
| LoRA | 모듈 + 헤드 | 약 0.4% |

In [ ]:
model.print_trainable_parameters()

- 부분 미세조정보다 학습 대상 증가 — 백본 내부까지 조정
- 전체 미세조정 대비 200분의 1 수준
- `r` 값 조정으로 학습 규모 변경 가능

---
## 5. 학습 실행

- 학습 절차는 앞 실습과 **동일**
- `learning_rate` — 헤드만 학습할 때보다 **낮게 설정**
- 근거 — 백본 내부까지 조정하므로 급격한 변화 방지
- 그 외 항목은 동일하게 유지하여 조건 통제

In [ ]:
args = TrainingArguments(
    output_dir="food_lora",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-4,
    remove_unused_columns=False,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
)

trainer.train()

---
## 6. 결과 확인

### 6-1. 정확도 측정

- 앞 실습과 **동일한 함수**로 측정 — 동일 기준 비교

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def accuracy(model, split="test", n=None):
    model.eval().to(device)
    data = ds[split]
    n = n or len(data)
    correct = 0
    with torch.no_grad():
        for i in range(n):
            item = data[i]
            x = item["pixel_values"].unsqueeze(0).to(device)
            pred = model(x).logits.argmax(-1).item()
            if pred == item["labels"]:
                correct += 1
    return correct / n

acc_lora = accuracy(model)
print("LoRA 정확도:", round(acc_lora, 4))

---
## 7. 저장

- 추가한 모듈만 저장 — 원본 백본은 공유
- 저장 용량이 매우 작음 — 용도별 모듈 교체 가능

In [ ]:
model.save_pretrained("food_lora")
processor.save_pretrained("food_lora")

import os
files = os.listdir("food_lora")
size = sum(os.path.getsize(os.path.join("food_lora", f)) for f in files)

print(files)
print("저장 용량:", round(size / 1024 / 1024, 2), "MB")

**재사용**

- 저장한 모듈을 원본 모델에 다시 결합하여 사용
- `PeftModel.from_pretrained(원본모델, "food_lora")`

---
## 8. 정리

**수행 내용**

- 원본 파라미터를 유지한 채 소형 모듈을 삽입하여 학습
- 학습 대상 — 전체의 약 0.4%
- 저장 대상 — 추가분만, 수 MB 수준

**확인한 사항**

- 백본 내부까지 조정 가능 — 부분 미세조정보다 성능 향상 여지
- 원본은 변경되지 않음 — 사전학습 표현 보존
- 용도별 모듈을 교체하며 사용 가능

**판단 기준**

- 부분 미세조정으로 성능이 부족한 경우 LoRA 적용 검토
- 원칙 — 가벼운 방식부터 시도, 필요 시 범위 확대

**참고**

- PEFT에는 이 외에도 어댑터 · 프롬프트 튜닝 등 다수 존재
- 라이브러리 — `peft`

---
